# Rollout training, two stages

Implements `writeup/rollout_plan.md`. Stage 1 trains one **base checkpoint** (teacher forcing + `mc` +
consistency with the `all` objective on recorded rows; threshold conditioning, RoPE, ordinal MODE tokens).
Stage 2 restarts from it with **proposal rows** added to the consistency batch: the model's own
reward-conditioned rollouts from recorded prefixes (`tau ~ U[0, length)`), the request chosen by the
**quantile rule** (the most ambitious threshold the model still believes with probability `p`), at most
`MAX_STEPS` imagined steps with NOR dynamics and learned END, no filtering, and the residual enforced on the
imagined segment only. Every arm is a dict; a **control arm** continues the base objective from the same
restart, and improvements are read against it.

Nothing imagined is ever fitted as data and no maze structure enters training. The real maze appears only in
the evaluation callbacks and in the `rollout_eval/*` check of each refresh's buffer (invalid imagined
transitions by kind), which is the world-model curve every arm must keep flat.

In [ ]:
REPO = "https://github.com/amdson/scrl.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
USE_DRIVE = True  #@param {type:"boolean"}
RUNS_DIR = "/content/drive/MyDrive/sillyrl/runs"  #@param {type:"string"}
USE_WANDB = True  #@param {type:"boolean"}
WANDB_PROJECT = "sillyrl-rollouts"  #@param {type:"string"}


In [ ]:
# Clone (or update) the repo on Colab. Locally (SMOKE): run from the repo root with tiny settings.
import os, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
SMOKE = not IN_COLAB                       # local execution = a fast smoke test of every cell

if IN_COLAB:
    url = REPO
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        if token:
            url = REPO.replace("https://", f"https://{token}@")
    except Exception:
        pass  # no secret: public repo
    if not os.path.exists("/content/sillyrl/.git"):
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/sillyrl"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/sillyrl", "pull", "-q"], check=True)
    os.chdir("/content/sillyrl")
    sys.path.insert(0, "/content/sillyrl")
    # Colab's preinstalled flax can lag its JAX; upgrade flax and optax, and keep the CUDA plugin in step with JAX.
    import importlib.metadata as md
    jax_before = md.version("jax")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "flax", "optax"], check=True)
    jax_after = md.version("jax")
    if jax_after != jax_before:
        plugins = sorted({d.metadata["Name"] for d in md.distributions()
                          if (d.metadata["Name"] or "").lower().replace("_", "-").startswith("jax-cuda")})
        if plugins:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={jax_after}" for p in plugins]], check=True)
        print(f"jax {jax_before} -> {jax_after}; plugins updated: {plugins}")
else:
    while not os.path.exists("maze_consistency") and os.getcwd() != "/":
        os.chdir("..")
    sys.path.insert(0, os.getcwd())
    USE_DRIVE, USE_WANDB = False, False

import jax, flax, optax
print("commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("jax", jax.__version__, "| flax", flax.__version__, "| optax", optax.__version__, "|", jax.devices(), "| smoke:", SMOKE)


In [ ]:
# Where runs are saved. Must be set before importing maze_consistency.train.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["RUNS_DIR"] = RUNS_DIR
else:
    os.environ["RUNS_DIR"] = "runs/smoke" if SMOKE else "runs"
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
print("runs ->", os.environ["RUNS_DIR"])


In [ ]:
# Dataset (100k random walks; deterministic) and the exact test set for this binning and conditioning.
MAZE_KW = dict(binning="geometric", n_bins=24)
COND = "threshold"
TEST_PATH = f"data/canonical/testset_geo24_{COND}.npz"
subprocess.run([sys.executable, "run.py", "dataset"], check=True, capture_output=True)
from maze_consistency.dataset import canonical_maze
from maze_consistency.testset import build_testset
_maze = canonical_maze(**MAZE_KW)
print(_maze, "| empty bins (dead classes, skipped by the test set):", _maze.empty_bins)
if not os.path.exists(TEST_PATH):
    build_testset(_maze, path=TEST_PATH, cond=COND)


In [ ]:
# Weights & Biases (Colab secret WANDB_API_KEY). Panel sections:
#   loss/*        tf, mc, cons (unweighted per-row mean), cons_rec / cons_prop (the two halves), cons_w (the
#                 weighted term actually added to the loss)
#   diag/*        cond_gap, info_gain
#   act_kl/*, value_kl/*, cons/*, enrich/*, goalward/*, start/*   eval_fn metrics per checkpoint
#   sampler/*     per refresh: request quantile p, request histogram, prefix and endpoint log-belief in the
#                 request, their ratio (the propagation signal), fraction of flat rows, tau, END rate
#   gnorm/*, cons_rows/*   every GRAD_EVERY steps: each term's gradient norm alone; consistency split into
#                 recorded vs proposal rows (no clipping: this is the monitor)
#   rollout_eval/*  EVALUATION ONLY, real maze, on the sampler's latest buffer: invalid imagined transitions
#                 (total and by kind: wrong direction / wall / teleport), END at the goal, prefix distances
import numpy as np
from maze_consistency.evaluate import imagined_rollout_eval

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb
    try:
        from google.colab import userdata
        wandb.login(key=userdata.get("WANDB_API_KEY"))
    except Exception:
        wandb.login()


def buffer_checks(maze, sub):
    """Real-maze checks of a rollout sampler's latest buffer (evaluation only, nothing feeds back), plus the
    belief-ratio readout split by prefix distance. Returns a flat dict."""
    ro, tau, _ = sub.last_rollout()
    rd = sub.last_readout()
    out = {f"rollout_eval/{k}": v for k, v in imagined_rollout_eval(maze, ro, tau).items()}
    dist = maze.dist[ro["positions"][np.arange(len(tau)), tau].astype(np.int64)][rd["keep"]]
    ratio = rd["lb_end"] - rd["lb_prefix"]
    far = dist >= 10
    out["rollout_eval/prefix_dist_mean"] = float(dist.mean())
    out["rollout_eval/prefix_far_frac"] = float(far.mean())
    out["sampler/ratio_far"] = float(ratio[far].mean()) if far.any() else np.nan
    out["sampler/ratio_near"] = float(ratio[~far].mean()) if (~far).any() else np.nan
    out["_dist"], out["_ratio"], out["_req"] = dist, ratio, rd["req"]
    return out


def sampler_readouts(maze, sampler, seen):
    """Latest refresh of every late sampler as flat metrics; `seen` dedups the per-buffer checks."""
    out, arrays = {}, {}
    for j, sub in enumerate(getattr(sampler, "late", None) or []):
        if not sub.history:
            continue
        h = sub.history[-1]
        tag = f"sampler/{j}"
        for k in ("knob", "ended", "mean_imagined", "mean_tau", "lb_prefix", "lb_end", "ratio", "frac_flat"):
            if h.get(k) is not None:
                out[f"{tag}/{k}"] = float(h[k])
        share = h["bins"] / max(h["bins"].sum(), 1)
        out[f"{tag}/req_mean"] = float((share * np.arange(len(share))).sum())
        if hasattr(sub, "last_rollout") and h["step"] != seen.get(j):
            seen[j] = h["step"]
            chk = buffer_checks(maze, sub)
            arrays = {k: chk.pop(k) for k in list(chk) if k.startswith("_")}
            out.update(chk)
            sub.eval_history.append(dict(step=h["step"], **chk))
    return out, arrays


def make_logger(run, lc, sampler, config, maze):
    """metrics_fn for train(): train parts, eval metrics and the samplers' readouts to W&B. Also keeps the
    sampler readouts in memory (sub.eval_history) so the analysis cells work without W&B."""
    seen = {}
    for sub in getattr(sampler, "late", None) or []:
        sub.eval_history = []
    wb = None
    if USE_WANDB:
        run_id = "".join(ch if ch.isalnum() else "-" for ch in run)
        wb = wandb.init(project=WANDB_PROJECT, name=run, id=run_id, resume="allow",
                        config=dict(config, **{f"loss.{k}": v for k, v in lc.__dict__.items()}), reinit=True)

    def fn(step, m, kind):
        out = {}
        for k, v in m.items():
            if k == "step" or not np.isscalar(v):
                continue
            if kind == "train" and not k.startswith(("gnorm/", "cons_rows/")):
                out[f"loss/{k}" if k in ("tf", "mc", "td", "cons", "cons_w", "cons_rec", "cons_prop") else f"diag/{k}"] = float(v)
            else:
                out[k] = float(v)
        if kind == "train" and sampler is not None:
            more, arrays = sampler_readouts(maze, sampler, seen)
            out.update(more)
            if wb is not None and arrays:
                out["sampler/req_hist"] = wandb.Histogram(arrays["_req"])
                out["rollout_eval/prefix_dist_hist"] = wandb.Histogram(arrays["_dist"])
                out["sampler/ratio_hist"] = wandb.Histogram(arrays["_ratio"])
        if wb is not None:
            wb.log(out, step=step)

    fn.finish = (wb.finish if wb is not None else (lambda: None))
    return fn


## 1. Stage 1: the base checkpoint

Trained once under a name that records what it depends on; skipped when it exists. Every stage-2 arm starts
from it and is evaluated at step 0, so the shared baseline is in every arm's history.

In [ ]:
from maze_consistency.dataset import load as load_data
from maze_consistency.dp import compute_ground_truth
from maze_consistency.tokens import Tokenizer
from maze_consistency.model import ModelConfig, MazeTransformer, make_forward
from maze_consistency.testset import load_testset, stratified_rows, score
from maze_consistency.evaluate import make_enrichment_eval
from maze_consistency.train import train, load_run, LossConfig, RUNS_DIR, N_HELDOUT
from maze_consistency import probe
import maze_consistency.consistency as C

MAZE, DATA = load_data(**MAZE_KW)
TOK = Tokenizer(MAZE, cond=COND)
GT = compute_ground_truth(MAZE)
N_TRAIN = len(DATA["length"]) - N_HELDOUT
HELDOUT = np.random.default_rng(0).choice(np.arange(N_TRAIN, N_TRAIN + N_HELDOUT), 64, replace=False)
FAR = 10

# the base configuration: the current best arm (cos_3e-3_rope_ord_mongo of late_rollouts.ipynb) with `all`
OBJECTIVE, LAMBDA, CONS_BATCH = "all", 0.14, 16
BASE_KW = dict(d_model=256, n_layers=8, n_heads=4, lr=1e-3, warmup=450, cosine=True, batch=32,
               pos_enc="rope", mode_enc="ordinal")
BASE_STEPS = 5000
EVAL_EVERY, EVAL_PER_SETTING, ENRICH_N, LOG_EVERY, GRAD_EVERY, CKPT_EVERY = 500, 50, 128, 100, 100, 1000
if SMOKE:
    BASE_KW.update(d_model=32, n_layers=1, n_heads=2, warmup=5)
    BASE_STEPS = 40
    EVAL_EVERY, EVAL_PER_SETTING, ENRICH_N, LOG_EVERY, GRAD_EVERY, CKPT_EVERY = 20, 4, 16, 10, 10, 0
BASE_LC = LossConfig(mc=True, cons=True, cons_loss=OBJECTIVE, w_cons=LAMBDA, cons_batch=CONS_BATCH)
BASE_RUN = f"base/geo{MAZE.K}_{COND}_{BASE_KW['pos_enc']}_{BASE_KW['mode_enc']}_d{BASE_KW['d_model']}x{BASE_KW['n_layers']}_{OBJECTIVE}_{BASE_STEPS}"

TS = load_testset(TEST_PATH)
ROWS = stratified_rows(TS, EVAL_PER_SETTING, seed=0)
ENRICH = make_enrichment_eval(TOK, MAZE, DATA, n=ENRICH_N)
CELL_DIST = MAZE.dist[MAZE.start_cells]


def make_eval_fn(model):
    """Exact-test metrics, held-out consistency, enrichment, and the start-value probe (signed error and
    far-start KL of the NOR value head at prefix 0 against the exact random walk)."""
    cons_eval = C.make_heldout_eval(model, TOK, MAZE, DATA, HELDOUT)

    def eval_fn(params, fwd):
        m = score(params, fwd, TOK, TS, ROWS)
        m.pop("per_setting")
        m.update(cons_eval(params))
        m.update(ENRICH(params, fwd))
        sv = probe.start_values(fwd, params, TOK, MAZE, GT)
        far = sv["dist"] >= FAR
        kl = (sv["h0"] * (np.log(sv["h0"] + 1e-30) - np.log(sv["q"] + 1e-30))).sum(-1)
        m["start/abs_err"] = float(np.abs(sv["err"]).mean())
        m["start/abs_logratio_far"] = float(np.abs(sv["log_ratio"][far]).mean())
        m["start/kl_far"] = float(kl[far].mean())
        m["start/kl_near"] = float(kl[~far].mean())
        return m

    return eval_fn


def model_for(kw):
    cfg = ModelConfig.for_tokenizer(TOK, d_model=kw["d_model"], n_layers=kw["n_layers"], n_heads=kw["n_heads"],
                                    pos_enc=kw["pos_enc"], mode_enc=kw["mode_enc"])
    return MazeTransformer(cfg)


if os.path.exists(os.path.join(RUNS_DIR, BASE_RUN, "history.json")):
    print(f"[skip] {BASE_RUN} exists")
else:
    logger = make_logger(BASE_RUN, BASE_LC, None, dict(BASE_KW, stage=1, cond=COND, **MAZE_KW), MAZE)
    train(name=BASE_RUN, seed=0, steps=BASE_STEPS, loss=BASE_LC, eval_fn=make_eval_fn(model_for(BASE_KW)),
          eval_every=EVAL_EVERY, log_every=LOG_EVERY, metrics_fn=logger, grad_every=GRAD_EVERY,
          ckpt_every=CKPT_EVERY, maze_kw=MAZE_KW, cond=COND, **BASE_KW)
    logger.finish()
BASE_PARAMS, BASE_CFG = load_run(BASE_RUN)
print("base:", BASE_RUN)


## 2. Stage 2: the rollout arms

Each arm is a dict of sampler and loss settings, trained from `BASE_PARAMS` with a fresh optimizer and a short
warmup. `control` adds no proposals. The proposal rows' weight `w_prop` starts at a tenth of `LAMBDA`; the
`gnorm/cons_proposal` panel against `gnorm/cons_recorded` says whether it can go up. `p` is a float or a
callable of the step (the schedule).

In [ ]:
ARM_STEPS = 3000
ARM_KW = dict(BASE_KW, lr=5e-4, warmup=100, cosine=True)
if SMOKE:
    ARM_STEPS, ARM_KW = 30, dict(BASE_KW, warmup=3)
KEEP_RECORDED = 0.5
MAX_STEPS = 10
BUFFER_N, REFRESH_EVERY = (256, 250) if not SMOKE else (32, 10)


def p_schedule(p0=0.1, p1=1e-3, steps=ARM_STEPS):
    """Log-linear from p0 to p1 over the arm."""
    return lambda step: float(p0 * (p1 / p0) ** min(max(step / steps, 0.0), 1.0))


# name -> dict(request, p | beta, w_prop, keep_recorded, max_steps, cons_batch); missing keys take the defaults.
# Round 1 (w_prop = 0.1 * LAMBDA) showed proposal rows with a raw gradient norm equal to the recorded rows' and
# a residual 2-3x theirs, rising as p fell, while every metric sat on control: the applied weight was ~3% of
# the step. Round 2 raises the weight at a fixed ambitious p, then varies the batch and the request.
ARMS = {
    "control":        dict(proposals=False),
    # round 1
    "quantile_p0.1":  dict(request="quantile", p=0.1, w_prop=0.1 * LAMBDA),
    "quantile_sched": dict(request="quantile", p=p_schedule(), w_prop=0.1 * LAMBDA),
    # round 2: weight
    "p1e-3_w1":       dict(request="quantile", p=1e-3, w_prop=LAMBDA),
    "p1e-3_w10":      dict(request="quantile", p=1e-3, w_prop=10 * LAMBDA),
    # round 2: variance -- twice the consistency batch (same recorded share, 16 proposal rows per step)
    "p1e-3_w1_cb32":  dict(request="quantile", p=1e-3, w_prop=LAMBDA, cons_batch=2 * CONS_BATCH),
    # round 2: request ambition at the working weight
    "p1e-5_w1":       dict(request="quantile", p=1e-5, w_prop=LAMBDA),
    "sched_w1":       dict(request="quantile", p=p_schedule(1e-2, 1e-5), w_prop=LAMBDA),
    # round 2: the tilt as the alternative request rule
    "tilt_b10_w1":    dict(request="tilt", beta=10.0, w_prop=LAMBDA),
}
ROUND = {k: ARMS[k] for k in ("control", "p1e-3_w1", "p1e-3_w10", "p1e-3_w1_cb32", "p1e-5_w1", "sched_w1", "tilt_b10_w1")}
if SMOKE:
    ROUND = {k: ARMS[k] for k in ("control", "p1e-3_w10", "p1e-3_w1_cb32")}

PREFIX = f"arms/{os.path.basename(BASE_RUN)}"
SAMPLERS = {}


def run_arm(name, arm, seed=0, steps=ARM_STEPS, skip_existing=True):
    run = f"{PREFIX}/{name}_s{seed}"
    if skip_existing and os.path.exists(os.path.join(RUNS_DIR, run, "history.json")):
        print(f"[skip] {run} exists")
        return
    model = model_for(ARM_KW)
    lc = BASE_LC if not arm.get("proposals", True) else \
        LossConfig(mc=True, cons=True, cons_loss=OBJECTIVE, w_cons=LAMBDA, cons_batch=arm.get("cons_batch", CONS_BATCH),
                   w_prop=arm.get("w_prop", -1.0))
    sampler = None
    if arm.get("proposals", True):
        roll = C.make_rollout_sampler(model, TOK, MAZE, DATA, N_TRAIN, request=arm.get("request", "quantile"),
                                      p=arm.get("p", 0.1), beta=arm.get("beta", 0.0), buffer_n=BUFFER_N,
                                      refresh_every=REFRESH_EVERY, max_steps=arm.get("max_steps", MAX_STEPS))
        sampler = C.make_phased_sampler(TOK, MAZE, DATA, N_TRAIN, late=[roll], start=0,
                                        keep_recorded=arm.get("keep_recorded", KEEP_RECORDED))
        sampler.late = [roll]
    SAMPLERS[run] = sampler
    cfg = {k: (v if np.isscalar(v) or isinstance(v, str) else "schedule") for k, v in arm.items()}
    logger = make_logger(run, lc, sampler, dict(ARM_KW, stage=2, base=BASE_RUN, cond=COND, **cfg, **MAZE_KW), MAZE)
    train(name=run, seed=seed, steps=steps, loss=lc, eval_fn=make_eval_fn(model), eval_every=EVAL_EVERY,
          log_every=LOG_EVERY, metrics_fn=logger, grad_every=GRAD_EVERY, ckpt_every=CKPT_EVERY,
          cons_sampler=sampler, init_params=BASE_PARAMS, maze_kw=MAZE_KW, cond=COND, **ARM_KW)
    logger.finish()


for name, arm in ROUND.items():
    run_arm(name, arm)


## 3. Curves

Test-side: far-start value KL and start-value error (the probe), enrichment, held-out consistency, and the
conditioned action KL on the far settings. Train-side: teacher forcing (must stay flat), the two consistency
halves, and the gradient norms. Step 0 of every arm is the base checkpoint.

In [ ]:
import json
import matplotlib.pyplot as plt


def load_history(prefix, which="test"):
    root = os.path.join(RUNS_DIR, prefix)
    out = {}
    for d in sorted(os.listdir(root)) if os.path.isdir(root) else []:
        p = os.path.join(root, d, "history.json")
        if os.path.exists(p):
            with open(p) as f:
                h = json.load(f)
            out.setdefault(d.rsplit("_s", 1)[0], []).append(h[which])
    return out


def plot_keys(prefix, keys, which="test", logy=True, figsize=(4.2, 3.3), order=None):
    runs = load_history(prefix, which)
    names = [n for n in (order or list(ARMS)) if n in runs] or list(runs)
    fig, ax = plt.subplots(1, len(keys), figsize=(figsize[0] * len(keys), figsize[1]), squeeze=False)
    for j, key in enumerate(keys):
        a = ax[0, j]
        for c, name in enumerate(names):
            hists = runs[name]
            steps = [m["step"] for m in hists[0]]
            ys = np.array([[m.get(key, np.nan) for m in h] for h in hists], dtype=float)
            if np.isnan(ys).all():
                continue
            a.plot(steps, np.nanmean(ys, 0), color=f"C{c}", label=name, marker="." if len(steps) < 12 else None)
        if logy:
            a.set_yscale("log")
        a.set_title(key, fontsize=9); a.set_xlabel("step"); a.grid(alpha=.3)
    ax[0, 0].legend(fontsize=7)
    fig.tight_layout()
    return fig


_best_at = lambda dd: int(MAZE.best_bin(np.flatnonzero(MAZE.dist == dd)[0]))
FAR_SETTINGS = [f"bin {_best_at(20)}", f"bin {_best_at(10)}", "best far"]
plot_keys(PREFIX, ["start/kl_far", "start/abs_logratio_far", "enrich/points", "cons/all"], logy=False); plt.show()
plot_keys(PREFIX, [f"act_kl/{s}" for s in FAR_SETTINGS] + [f"value_kl/{s}" for s in FAR_SETTINGS[:2]]); plt.show()
plot_keys(PREFIX, ["tf", "cons_rec", "cons_prop", "cond_gap"], which="train", logy=False); plt.show()
plot_keys(PREFIX, ["gnorm/tf", "gnorm/cons_recorded", "gnorm/cons_proposal"], which="train"); plt.show()


## 4. What the sampler proposed, and what the world model did

Per refresh, from the samplers of runs trained in this session: the request quantile, the belief in the
request at the prefix and at the endpoint (log), their ratio split by prefix distance, the request histogram,
and the invalid-transition curve by kind. A ratio near zero means the row is redundant with recorded rows; a
rising invalid-step fraction means the arm is breaking the world model.

In [ ]:
def sampler_curves(run):
    s = SAMPLERS.get(run)
    if s is None or not s.late or not s.late[0].history:
        print(run, ": no sampler readouts in this session"); return
    sub = s.late[0]
    h, e = sub.history, sub.eval_history
    st = [x["step"] for x in h]
    fig, ax = plt.subplots(1, 4, figsize=(17, 3.4))
    ax[0].plot(st, [x["lb_prefix"] for x in h], label="log q at prefix"); ax[0].plot(st, [x["lb_end"] for x in h], label="log q at endpoint")
    ax[0].set_title("belief in the request (mean log)", fontsize=9); ax[0].legend(fontsize=7)
    ax[1].plot(st, [x["ratio"] for x in h], label="all"); ax[1].plot([x["step"] for x in e], [x["sampler/ratio_far"] for x in e], label="far prefix")
    ax[1].plot([x["step"] for x in e], [x["sampler/ratio_near"] for x in e], label="near prefix"); ax[1].axhline(0, color="k", lw=.8, ls=":")
    ax[1].set_title("log belief ratio endpoint / prefix", fontsize=9); ax[1].legend(fontsize=7)
    bins = np.stack([x["bins"] for x in h]); ax[2].imshow(bins.T / np.maximum(bins.sum(1), 1)[None], aspect="auto", origin="lower", extent=[st[0], st[-1], 0, MAZE.K])
    ax[2].set_title("request histogram over refreshes", fontsize=9); ax[2].set_ylabel("threshold k")
    for k in ("invalid_step_frac", "wrong_dir_frac", "wall_frac", "teleport_frac"):
        ax[3].plot([x["step"] for x in e], [x[f"rollout_eval/{k}"] for x in e], label=k)
    ax[3].set_title("invalid imagined transitions (real maze, eval only)", fontsize=9); ax[3].legend(fontsize=7); ax[3].set_yscale("symlog", linthresh=1e-4)
    for a in ax: a.set_xlabel("step"); a.grid(alpha=.3)
    fig.suptitle(run, fontsize=9); fig.tight_layout()
    print(f"{run}: p {h[-1]['knob']}, END rate {np.mean([x['ended'] for x in h]):.2f}, imagined steps {np.mean([x['mean_imagined'] for x in h]):.1f}, "
          f"flat rows {np.mean([x['frac_flat'] for x in h]):.2f}, invalid steps {np.mean([x['rollout_eval/invalid_step_frac'] for x in e]):.4f}")
    return fig


for name in ROUND:
    sampler_curves(f"{PREFIX}/{name}_s0")
plt.show()
